# DBSCAN yoga subpose clustering

This notebook learns subposes from the training set and assigns both training and test samples. Requested subtype counts are: downdog 1, goddess 3, plank 4, tree 3, and warrior2 1.

Tree and warrior2 also receive a left/right `pose_side`, defined as the side with the more bent knee (the smaller knee angle). This side is included in `subpose_label` but is separate from the requested DBSCAN subtype count.

## Method

DBSCAN discovers density clusters rather than accepting a cluster count. For poses needing multiple subtypes, this notebook searches deterministic `min_samples` and `eps` candidates and chooses the exact-count result with the lowest training noise rate, breaking ties with silhouette score. Noise samples and test samples are assigned to the nearest learned cluster centroid so every row receives a usable subtype.

The scaler, PCA projection, DBSCAN tuning, and centroids are fitted only on training rows. Test rows never influence the learned clusters.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances_argmin, pairwise_distances_argmin_min, silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TARGET_SUBPOSES = {
    'downdog': 1,
    'goddess': 3,
    'plank': 4,
    'tree': 3,
    'warrior2': 1,
}
SIDE_AWARE_POSES = {'tree', 'warrior2'}
MIN_SAMPLES_CANDIDATES = range(3, 11)
EPS_QUANTILES = np.linspace(0.01, 0.99, 180)


def find_project_root(start: Path = Path.cwd()) -> Path:
    for directory in (start.resolve(), *start.resolve().parents):
        if (directory / 'csv_data' / 'keypoints_train_body_angles.csv').is_file():
            return directory
    raise FileNotFoundError('Could not find the body-angle CSV files.')


PROJECT_ROOT = find_project_root()
CSV_DIR = PROJECT_ROOT / 'csv_data'
INPUT_FILES = {
    split: CSV_DIR / f'keypoints_{split}_body_angles.csv'
    for split in ('train', 'test')
}
OUTPUT_FILES = {
    split: CSV_DIR / f'keypoints_{split}_dbscan_subposes.csv'
    for split in ('train', 'test')
}

PROJECT_ROOT

WindowsPath('C:/Users/vgohu/Desktop/yoga_project')

In [2]:
def feature_columns(dataframe: pd.DataFrame) -> list[str]:
    return [
        column for column in dataframe.columns
        if column.startswith('kp_') or column.endswith('_angle_deg')
    ]


def cluster_count(labels: np.ndarray) -> int:
    return len(set(labels)) - int(-1 in labels)


def fit_preprocessor(features: np.ndarray):
    scaler = StandardScaler()
    scaled = scaler.fit_transform(features)
    pca = PCA(n_components=0.95, svd_solver='full', random_state=RANDOM_STATE)
    projected = pca.fit_transform(scaled)
    return scaler, pca, projected


def tune_dbscan(projected: np.ndarray, target_clusters: int):
    candidates = []
    for min_samples in MIN_SAMPLES_CANDIDATES:
        neighbor_distances = NearestNeighbors(n_neighbors=min_samples).fit(projected).kneighbors()[0]
        kth_distances = np.sort(neighbor_distances[:, -1])
        eps_values = np.unique(np.quantile(kth_distances, EPS_QUANTILES))

        for eps in eps_values:
            if not np.isfinite(eps) or eps <= 0.0:
                continue
            model = DBSCAN(eps=float(eps), min_samples=min_samples).fit(projected)
            labels = model.labels_
            if cluster_count(labels) != target_clusters:
                continue

            non_noise = labels >= 0
            score = (
                silhouette_score(projected[non_noise], labels[non_noise])
                if target_clusters > 1 and len(np.unique(labels[non_noise])) > 1
                else 0.0
            )
            noise_rate = float((labels == -1).mean())
            candidates.append((noise_rate, -score, min_samples, float(eps), model))

    if not candidates:
        raise RuntimeError(
            f'DBSCAN could not produce exactly {target_clusters} clusters. '
            'Expand the min_samples or eps search grid.'
        )
    return min(candidates, key=lambda candidate: candidate[:4])


def stable_cluster_mapping(projected: np.ndarray, raw_labels: np.ndarray) -> dict[int, int]:
    old_labels = sorted(set(raw_labels) - {-1})
    centroids = {label: projected[raw_labels == label].mean(axis=0) for label in old_labels}
    ordered = sorted(old_labels, key=lambda label: tuple(centroids[label]))
    return {old_label: new_label for new_label, old_label in enumerate(ordered)}


def remap_labels(labels: np.ndarray, mapping: dict[int, int]) -> np.ndarray:
    return np.array([mapping.get(int(label), -1) for label in labels], dtype=int)


def cluster_centroids(projected: np.ndarray, labels: np.ndarray, count: int) -> np.ndarray:
    return np.vstack([projected[labels == label].mean(axis=0) for label in range(count)])


def assign_noise_to_centroids(
    projected: np.ndarray, labels: np.ndarray, centroids: np.ndarray
) -> np.ndarray:
    assigned = labels.copy()
    noise = assigned == -1
    if noise.any():
        assigned[noise] = pairwise_distances_argmin(projected[noise], centroids)
    return assigned


def bent_leg_side(dataframe: pd.DataFrame) -> pd.Series:
    return pd.Series(
        np.where(
            dataframe['left_knee_angle_deg'] < dataframe['right_knee_angle_deg'],
            'left',
            'right',
        ),
        index=dataframe.index,
        dtype='string',
    )


In [3]:
datasets = {split: pd.read_csv(path) for split, path in INPUT_FILES.items()}
features = feature_columns(datasets['train'])
if datasets['train'][features].isna().any().any() or datasets['test'][features].isna().any().any():
    raise ValueError('Feature columns contain missing values. Run the angle-imputation notebook first.')

results = {split: dataframe.copy() for split, dataframe in datasets.items()}
for result in results.values():
    result['subpose_id'] = -1
    result['pose_side'] = 'not_applicable'
    result['dbscan_was_noise'] = False
    result['subpose_label'] = ''

reports = []
for pose, target_count in TARGET_SUBPOSES.items():
    train_mask = datasets['train']['label'].eq(pose)
    test_mask = datasets['test']['label'].eq(pose)
    train_values = datasets['train'].loc[train_mask, features].to_numpy(dtype=float)
    test_values = datasets['test'].loc[test_mask, features].to_numpy(dtype=float)
    scaler, pca, train_projected = fit_preprocessor(train_values)
    test_projected = pca.transform(scaler.transform(test_values))

    if target_count == 1:
        train_raw = np.zeros(len(train_projected), dtype=int)
        test_raw = np.zeros(len(test_projected), dtype=int)
        train_assigned = train_raw.copy()
        test_assigned = test_raw.copy()
        noise_rate = 0.0
        min_samples = np.nan
        eps = np.nan
    else:
        noise_rate, negative_silhouette, min_samples, eps, model = tune_dbscan(
            train_projected, target_count
        )
        mapping = stable_cluster_mapping(train_projected, model.labels_)
        train_raw = remap_labels(model.labels_, mapping)
        centroids = cluster_centroids(train_projected, train_raw, target_count)
        train_assigned = assign_noise_to_centroids(train_projected, train_raw, centroids)

        core_labels = remap_labels(model.labels_[model.core_sample_indices_], mapping)
        nearest_core, nearest_distance = pairwise_distances_argmin_min(
            test_projected, model.components_
        )
        test_raw = np.where(nearest_distance <= eps, core_labels[nearest_core], -1)
        test_assigned = assign_noise_to_centroids(test_projected, test_raw, centroids)

    results['train'].loc[train_mask, 'subpose_id'] = train_assigned + 1
    results['test'].loc[test_mask, 'subpose_id'] = test_assigned + 1
    results['train'].loc[train_mask, 'dbscan_was_noise'] = train_raw == -1
    results['test'].loc[test_mask, 'dbscan_was_noise'] = test_raw == -1

    reports.append({
        'pose': pose,
        'requested_subposes': target_count,
        'training_samples': int(train_mask.sum()),
        'test_samples': int(test_mask.sum()),
        'min_samples': min_samples,
        'eps': eps,
        'training_noise_rate': noise_rate,
    })

for split, result in results.items():
    side_mask = result['label'].isin(SIDE_AWARE_POSES)
    result.loc[side_mask, 'pose_side'] = bent_leg_side(result.loc[side_mask])
    result['subpose_id'] = result['subpose_id'].astype(int)
    result['subpose_label'] = np.where(
        side_mask,
        result['label'] + '_' + result['pose_side'] + '_subpose_' + result['subpose_id'].astype(str),
        result['label'] + '_subpose_' + result['subpose_id'].astype(str),
    )
    result.to_csv(OUTPUT_FILES[split], index=False)

clustering_report = pd.DataFrame(reports)
clustering_report

C:\Users\vgohu\Anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\vgohu\Anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 282, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


,pose,requested_subposes,training_samples,test_samples,min_samples,eps,training_noise_rate
0,downdog,1,199,94,NaN,NaN,0.000000
1,goddess,3,172,80,3.0,10.630321,0.029070
2,plank,4,263,115,5.0,6.968052,0.057034
3,tree,3,159,69,3.0,7.936945,0.075472
4,warrior2,1,249,107,NaN,NaN,0.000000


In [4]:
for split, result in results.items():
    assert len(result) == len(datasets[split])
    assert (result['subpose_id'] > 0).all()
    assert result['subpose_label'].ne('').all()
    assert OUTPUT_FILES[split].is_file()
    for pose, requested_count in TARGET_SUBPOSES.items():
        pose_ids = result.loc[result['label'].eq(pose), 'subpose_id']
        assert pose_ids.between(1, requested_count).all()
        if split == 'train':
            assert pose_ids.nunique() == requested_count

subpose_summary = (
    pd.concat(
        [
            result.groupby(['label', 'pose_side', 'subpose_id']).size().rename(split)
            for split, result in results.items()
        ],
        axis=1,
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)
subpose_summary

,label,pose_side,subpose_id,train,test
0,downdog,not_applicable,1,199,94
1,goddess,not_applicable,1,4,0
2,goddess,not_applicable,2,162,78
3,goddess,not_applicable,3,6,2
4,plank,not_applicable,1,13,1
5,plank,not_applicable,2,124,44
6,plank,not_applicable,3,8,2
7,plank,not_applicable,4,118,68
8,tree,left,1,4,0
9,tree,left,2,50,29


## Outputs

- `csv_data/keypoints_train_dbscan_subposes.csv`
- `csv_data/keypoints_test_dbscan_subposes.csv`

Added columns: `subpose_id`, `pose_side`, `dbscan_was_noise`, and `subpose_label`. Review `dbscan_was_noise` and the displayed cluster sizes when assessing whether DBSCAN found meaningful density groups.